# PrecisionSkin - Eczema v2 pipeline  (run the cells top to bottom)

Everything for the eczema extension in **one notebook**. Each step is one cell; run them in order.

| Step | What it does | Time |
|---|---|---|
| 0 | Settings + checks | seconds |
| A | Reads **your finished review**, measures the drafts, builds the new source `eczema_corrected.yolov8` | 1 min |
| B | **Backs up** the old audit, re-runs the audit with the new source | 5-10 min |
| C | Builds **dataset v2** (same validation/test photos as v1), **trains**, evaluates, exports ONNX | about 2 h |
| D | **Compares v1 vs v2** on the same test photos | seconds |
| E | (optional) copies the new model into the app's `Model` folder | seconds |

**Safety**
- Your originals, the v1 dataset, the v1 runs and the v1 model are never changed. v2 gets its own folders (`dataset_v2`, `detector_v2_...`, `lesion_detector_v2_...`).
- The old audit results are copied to `localization_dataset/01_audit_reports_BACKUP_...` before the audit is re-run.
- Steps B and C run in **separate processes**, so memory is freed between steps (this avoids the out-of-memory crashes seen before). Their full logs are saved in `~/PrecisionSkinV1/pipeline_logs/`.
- Use the **`.venv_det`** kernel (it has PyTorch + ultralytics).
- Plug the laptop in, keep it awake, keep the Ubuntu window open, close Chrome. Step C is the long one.

Re-running is safe: a step that already finished tells you so and skips (or you can switch it off in Step 0).

## Step 0 - Settings  (edit this cell only)

In [ ]:
import os, sys, json, shutil, time, subprocess, hashlib, ast
from pathlib import Path
import pandas as pd

HOME = Path.home() / "PrecisionSkinV1"
REPO = Path("/mnt/c/Users/Bautista/source/repos/SkinDevApp")                       # latest clean notebooks + the app
SOURCES_ROOT = Path("/mnt/c/Users/Bautista/Downloads/Localization Training Datasets")
REVIEW_DIR = Path("/mnt/c/Users/Bautista/Documents/PrecisionSkin_EczemaReview")     # made by the draft notebook, finished by you
NEW_SOURCE_NAME = "eczema_corrected.yolov8"                                         # name of the new dataset folder (also used to keep it out of val/test)

NB_AUDIT = REPO / "PrecisionSkin_Localization_Dataset_Audit.ipynb"
NB_EXPORT = REPO / "PrecisionSkin_Detector_Export_and_Train.ipynb"
WORK = HOME / "detector_work"
AUDIT_DIR = HOME / "localization_dataset" / "01_audit_reports"
V1_JSON = WORK / "export" / "lesion_detector_v1.json"
V1_MANIFEST = WORK / "dataset_v1" / "dataset_manifest.csv"
LOG_DIR = HOME / "pipeline_logs"
APP_MODEL_DIR = REPO / "SkinDevApp" / "Model"

# ---- which steps to run -----------------------------------------------------------------------------------------
DO_BUILD_ECZEMA_SOURCE = True      # Step A
DO_AUDIT = True                    # Step B
DO_TRAIN_V2 = True                 # Step C  (the long one)
DO_COMPARE = True                  # Step D
COPY_MODEL_TO_APP = False          # Step E  (True = copy the v2 ONNX/JSON over the app's lesion_detector_v1 files)

MIN_REVIEWED_FRACTION = 0.90       # Step A refuses to build the dataset if fewer images than this were reviewed
OVERWRITE_ECZEMA_SOURCE = False    # False = if the new source folder already exists, leave it alone

# ---- training (same recipe as v1, so the comparison isolates the new data) --------------------------------------
MODEL = "yolov8n.pt"
IMGSZ = 640
EPOCHS = 100
PATIENCE = 25
SEEDS = [42]                       # [42, 7, 123] = three runs to report the spread for eczema (3x longer)
DATASET_VERSION = "v2"

LOG_DIR.mkdir(parents=True, exist_ok=True)
print("kernel python :", sys.executable)
try:
    import torch, ultralytics
    print("torch %s | ultralytics %s | GPU: %s" % (torch.__version__, ultralytics.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE"))
except Exception as e:
    print("!! this kernel cannot train (%s). Select the .venv_det kernel." % e)
checks = {"audit notebook": NB_AUDIT.exists(), "export notebook": NB_EXPORT.exists(), "review folder": REVIEW_DIR.exists(),
          "v1 model json": V1_JSON.exists(), "v1 manifest": V1_MANIFEST.exists(), "old audit master": (AUDIT_DIR / "master_metadata.csv").exists()}
for k, v in checks.items():
    print("%-18s %s" % (k, "OK" if v else "MISSING  <-- fix before continuing"))
assert all(checks.values()), "Something required is missing (see above)."

## Helper - run another notebook in a separate process
(Defines `run_notebook(...)`. Nothing runs yet.) It applies a few **exact text replacements** to the notebook's code cells (and refuses if a replacement does not match exactly once),
runs the cells in a fresh Python process, shows the important lines live and saves the full log.

In [ ]:
def run_notebook(nb_path, replacements, step, dry_run=False):
    """Run all code cells of nb_path in a fresh process. replacements = [(old, new), ...]; each must match exactly once."""
    nb = json.load(open(nb_path, encoding="utf-8"))
    cells = ["".join(c["source"]) for c in nb["cells"] if c["cell_type"] == "code"]
    cells = [c for c in cells if not c.lstrip().startswith("%pip")]
    counts = [0] * len(replacements)
    patched = []
    for c in cells:
        for i, (old, new) in enumerate(replacements):
            n = c.count(old)
            if n:
                c = c.replace(old, new); counts[i] += n
        patched.append(c)
    bad = [(replacements[i][0][:60], counts[i]) for i in range(len(replacements)) if counts[i] != 1]
    if bad:
        raise RuntimeError("These settings were not found exactly once in %s (the notebook changed?): %s" % (Path(nb_path).name, bad))
    for i, c in enumerate(patched):
        ast.parse(c)                                     # every patched cell must still be valid Python
    print("[%s] %d cells ready, %d settings applied." % (step, len(patched), len(replacements)))
    if dry_run:
        return None

    script = LOG_DIR / ("%s_runner.py" % step)
    script.write_text(
        "import os, sys, traceback\nos.environ['MPLBACKEND'] = 'Agg'\nG = {'__name__': '__main__'}\nCELLS = %r\n"
        "for i, src in enumerate(CELLS):\n    try:\n        exec(compile(src, 'cell%%d' %% i, 'exec'), G)\n"
        "    except SystemExit:\n        raise\n    except Exception:\n        print('!!! ERROR in cell', i); traceback.print_exc(); sys.exit(1)\nprint('ALL CELLS RAN')\n" % (patched,),
        encoding="utf-8")
    log = LOG_DIR / ("%s_%s.log" % (step, time.strftime("%Y%m%d_%H%M%S")))
    t0, last_prog, tail = time.time(), 0.0, []
    with open(log, "w", encoding="utf-8") as lf:
        p = subprocess.Popen([sys.executable, "-u", str(script)], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            line = line.rstrip("\n")
            lf.write(line + "\n"); tail.append(line); tail = tail[-40:]
            noisy = ("WARN" in line and "global" in line) or line.startswith("Requirement already") or not line.strip()
            is_bar = "%|" in line or "it/s" in line or "s/it" in line
            if is_bar:
                if time.time() - last_prog > 30:
                    print("   ..", line.strip()[:150]); last_prog = time.time()
            elif not noisy:
                print(line[:200])
        p.wait()
    mins = (time.time() - t0) / 60
    if p.returncode != 0:
        print("\n".join(tail[-15:]))
        raise RuntimeError("[%s] FAILED after %.1f min. Full log: %s" % (step, mins, log))
    print("[%s] finished OK in %.1f min. Full log: %s" % (step, mins, log))
    return log

print("helper ready")

## Step A - Build the new eczema source from your finished review
Reads every image **you saved** in the review tool (an image counts only if its label file was saved after the drafts were made), measures how good the machine drafts were
(a number for the paper), and writes the new source `eczema_corrected.yolov8` next to your other datasets. Images with no eczema box are left out.

In [ ]:
if not DO_BUILD_ECZEMA_SOURCE:
    print("Step A switched off.")
else:
    t_draft = json.load(open(REVIEW_DIR / "_drafts" / "meta.json"))["drafted_at"]
    asg = pd.read_csv(REVIEW_DIR / "review_assignments.csv")
    name_col = "review_name" if "review_name" in asg.columns else "filename"

    def read_boxes(p):
        out = []
        if Path(p).exists():
            for ln in Path(p).read_text().splitlines():
                t = ln.split()
                if len(t) == 5:
                    out.append(tuple(float(x) for x in t[1:]))
        return out

    def iou(a, b):
        ax0, ay0, ax1, ay1 = a[0] - a[2] / 2, a[1] - a[3] / 2, a[0] + a[2] / 2, a[1] + a[3] / 2
        bx0, by0, bx1, by1 = b[0] - b[2] / 2, b[1] - b[3] / 2, b[0] + b[2] / 2, b[1] + b[3] / 2
        iw, ih = max(0.0, min(ax1, bx1) - max(ax0, bx0)), max(0.0, min(ay1, by1) - max(ay0, by0))
        i = iw * ih; u = a[2] * a[3] + b[2] * b[3] - i
        return i / u if u > 0 else 0.0

    def match(dr, fin, thr=0.5):
        used, tp = set(), 0
        for d in dr:
            best, bj = 0.0, None
            for j, f in enumerate(fin):
                if j in used: continue
                v = iou(d, f)
                if v > best: best, bj = v, j
            if bj is not None and best >= thr:
                used.add(bj); tp += 1
        return tp

    rows = []
    for r in asg.itertuples():
        name = getattr(r, name_col); stem = Path(name).stem
        lp = REVIEW_DIR / r.reviewer / "labels" / (stem + ".txt")
        saved = lp.exists() and lp.stat().st_mtime > t_draft + 1
        fin, dr = read_boxes(lp), read_boxes(REVIEW_DIR / "_drafts" / (stem + ".txt"))
        rows.append({"name": name, "reviewer": r.reviewer, "reviewed": bool(saved), "n_draft": len(dr), "n_final": len(fin), "tp": match(dr, fin) if saved else 0})
    prog = pd.DataFrame(rows)
    rev = prog[prog.reviewed]
    frac = len(rev) / max(1, len(prog))
    print("REVIEWED (saved): %d of %d (%.0f%%)" % (len(rev), len(prog), 100 * frac))
    print("  no eczema box (X / all deleted): %d | included: %d | boxes after review: %d (machine had drafted %d)" % (
        int((rev.n_final == 0).sum()), int((rev.n_final > 0).sum()), int(rev.n_final.sum()), int(rev.n_draft.sum())))
    print("  DRAFT PRECISION (draft boxes you kept, IoU>=0.5): %.1f%%" % (100 * rev.tp.sum() / max(1, rev.n_draft.sum())))
    print("  DRAFT RECALL    (your final boxes the detector had): %.1f%%" % (100 * rev.tp.sum() / max(1, rev.n_final.sum())))
    assert frac >= MIN_REVIEWED_FRACTION, "Only %.0f%% reviewed - finish the review first (or lower MIN_REVIEWED_FRACTION)." % (100 * frac)

    target = SOURCES_ROOT / NEW_SOURCE_NAME
    if target.exists() and any(target.iterdir()) and not OVERWRITE_ECZEMA_SOURCE:
        print("\n%s already exists - left unchanged (set OVERWRITE_ECZEMA_SOURCE = True to rebuild it)." % NEW_SOURCE_NAME)
    else:
        if target.exists():
            shutil.rmtree(target)
        (target / "train" / "images").mkdir(parents=True); (target / "train" / "labels").mkdir(parents=True)
        use = rev[rev.n_final > 0]
        for r in use.itertuples():
            shutil.copy2(REVIEW_DIR / r.reviewer / "images" / r.name, target / "train" / "images" / r.name)
            shutil.copy2(REVIEW_DIR / r.reviewer / "labels" / (Path(r.name).stem + ".txt"), target / "train" / "labels" / (Path(r.name).stem + ".txt"))
        (target / "data.yaml").write_text("train: ../train/images\nval: ../train/images\nnc: 1\nnames: ['Eczema']\n")
        (target / "ORIGIN.txt").write_text("Boxes drafted by the PrecisionSkin detector (v1) and corrected by the researchers.\nOrigin: machine-drafted, human-corrected. TRAINING ONLY (never validation/test).\n")
        print("\nwritten: %s  (%d images)" % (target, len(use)))

## Step B - Re-run the audit with the new source
First **backs up** the current audit results, then runs the audit notebook in a separate process. The new source is declared facial (the images come from your classifier's facial folders).
The label `Eczema` maps by itself. Everything else behaves exactly as in the audit you already ran (cached features make it fast).

In [ ]:
BACKUP_DIR = None
if not DO_AUDIT:
    print("Step B switched off.")
else:
    # ---- 1) back up the old audit (the v1 split is defined by it; Step C needs the OLD master_metadata.csv) ----
    existing = sorted((AUDIT_DIR.parent).glob("01_audit_reports_BACKUP_*"))
    BACKUP_DIR = existing[0] if existing else AUDIT_DIR.parent / ("01_audit_reports_BACKUP_" + time.strftime("%Y%m%d_%H%M%S"))
    if not BACKUP_DIR.exists():
        shutil.copytree(AUDIT_DIR, BACKUP_DIR, ignore=shutil.ignore_patterns("features_cache.pkl"))
        print("old audit backed up to:", BACKUP_DIR)
    else:
        print("backup already exists (kept as it is):", BACKUP_DIR)
    assert (BACKUP_DIR / "master_metadata.csv").exists()

    # ---- 2) run the audit, declaring the new source facial ----
    already = False
    try:
        already = NEW_SOURCE_NAME in set(pd.read_csv(AUDIT_DIR / "master_metadata.csv", usecols=["source_dataset"])["source_dataset"])
    except Exception:
        pass
    if already:
        print("The audit already contains %s - nothing to do. (Delete the 'already' check in this cell to force a re-run.)" % NEW_SOURCE_NAME)
    else:
        run_notebook(NB_AUDIT, [("SOURCE_SITE_DECLARATION = {\n", 'SOURCE_SITE_DECLARATION = {\n    "%s": "facial",     # added by the eczema v2 pipeline\n' % NEW_SOURCE_NAME)], "B_audit")

    # ---- 3) what the audit says about the new source ----
    m = pd.read_csv(AUDIT_DIR / "master_metadata.csv", low_memory=False)
    s = m[m.source_dataset == NEW_SOURCE_NAME]
    print("\nnew source in the audit: %d images | decisions: %s | condition: %s" % (len(s), s.decision.value_counts().to_dict(), s.condition.value_counts().to_dict()))
    print("audit total: %d images | review queue: %d" % (len(m), int((m.decision == "review").sum())))

## Step C - Build dataset v2, train, evaluate, export  (about 2 hours)
Runs the export-and-train notebook in a separate process with these settings: `DATASET_VERSION = "v2"`, the **validation/test photos pinned to exactly the ones v1 used**
(so the comparison is fair), the new eczema source used for **training only**, and the training recipe from Step 0.
Progress lines appear here; the complete log is saved in `~/PrecisionSkinV1/pipeline_logs/`.
**Do not start anything else on the GPU while this runs.**

In [ ]:
if not DO_TRAIN_V2:
    print("Step C switched off.")
else:
    assert BACKUP_DIR is not None or any((AUDIT_DIR.parent).glob("01_audit_reports_BACKUP_*")), "Run Step B first (the old audit backup is needed)."
    backup = BACKUP_DIR or sorted((AUDIT_DIR.parent).glob("01_audit_reports_BACKUP_*"))[0]
    repl = [
        ('DATASET_VERSION = "v1"', 'DATASET_VERSION = "%s"' % DATASET_VERSION),
        ("PIN_SPLITS_FROM = None", 'PIN_SPLITS_FROM = {"manifest": Path(r"%s"), "master": Path(r"%s")}' % (V1_MANIFEST, backup / "master_metadata.csv")),
        ('MODEL = "yolov8n.pt"', 'MODEL = "%s"' % MODEL),
        ("IMGSZ = 640", "IMGSZ = %d" % IMGSZ),
        ("EPOCHS = 100", "EPOCHS = %d" % EPOCHS),
        ("PATIENCE = 25", "PATIENCE = %d" % PATIENCE),
        ("SEEDS = [42]", "SEEDS = %s" % (SEEDS,)),
    ]
    v2_json = WORK / "export" / ("lesion_detector_%s_%s_imgsz%d.json" % (DATASET_VERSION, Path(MODEL).stem, IMGSZ))
    if v2_json.exists():
        print("v2 already trained and exported:", v2_json, "\n(delete it to train again)")
    else:
        run_notebook(NB_EXPORT, repl, "C_export_train")

## Step D - Compare v1 and v2 on the SAME test photos
First checks that the test photos really are the same, then prints precision / recall / mAP per class for both models and the difference.
**Eczema has only 32 test photos**, so small differences there are noise; say so in the paper (or train with `SEEDS = [42, 7, 123]` and report the spread).

In [ ]:
if not DO_COMPARE:
    print("Step D switched off.")
else:
    v2_json = WORK / "export" / ("lesion_detector_%s_%s_imgsz%d.json" % (DATASET_VERSION, Path(MODEL).stem, IMGSZ))
    j1, j2 = json.load(open(V1_JSON)), json.load(open(v2_json))
    backup = sorted((AUDIT_DIR.parent).glob("01_audit_reports_BACKUP_*"))[0]

    # --- same test photos? (compare by source + original file name, because image ids change when the audit is re-run) ---
    def keys(manifest, master, split):
        man = pd.read_csv(manifest); ms = pd.read_csv(master, usecols=["image_id", "source_dataset", "original_filename"], low_memory=False)
        man = man[man.split == split][["image_id"]].merge(ms, on="image_id", how="left")
        return set(zip(man.source_dataset, man.original_filename))
    m2 = WORK / ("dataset_%s" % DATASET_VERSION) / "dataset_manifest.csv"
    for sp in ("val", "test"):
        a, b = keys(V1_MANIFEST, backup / "master_metadata.csv", sp), keys(m2, AUDIT_DIR / "master_metadata.csv", sp)
        print("%-5s photos: v1 %d | v2 %d | IDENTICAL SET: %s" % (sp, len(a), len(b), a == b))

    def table(j):
        r = next(iter(j["test_metrics_by_seed"].values()))
        out = {k: v for k, v in r["per_class"].items()}
        out["all"] = r["overall"]
        return out
    t1, t2 = table(j1), table(j2)
    rows = []
    for cls in ("acne", "hyperpigmentation", "eczema", "all"):
        for met in ("precision", "recall", "mAP50", "mAP50_95"):
            a, b = t1[cls][met], t2[cls][met]
            rows.append({"class": cls, "metric": met, "v1": round(a, 3), "v2": round(b, 3), "change": round(b - a, 3)})
    df = pd.DataFrame(rows)
    print("\nTEST-SET RESULTS (same photos):")
    print(df.pivot(index="metric", columns="class", values=["v1", "v2", "change"]).round(3).to_string())
    print("\ntest photos per class:", j2.get("test_photos_per_class"))
    print("v2 box origins:", j2.get("box_origin_counts"))
    print("v1 CPU latency:", j1.get("cpu_latency"), "| v2:", j2.get("cpu_latency"))
    df.to_csv(LOG_DIR / "v1_vs_v2_test_comparison.csv", index=False)
    print("\nsaved:", LOG_DIR / "v1_vs_v2_test_comparison.csv")

## Step E - (optional) put the v2 model into the app
Only if v2 is better. Copies the v2 ONNX and JSON over `SkinDevApp/Model/lesion_detector_v1.onnx` / `.json` (the names the app loads). The old v1 files stay in
`~/PrecisionSkinV1/detector_work/export/`. Afterwards: rebuild the app, and re-check the three confidence thresholds in Settings with the new model's `BoxF1_curve.png`.

In [ ]:
if not COPY_MODEL_TO_APP:
    print("Step E switched off (set COPY_MODEL_TO_APP = True in Step 0 if you want it).")
else:
    stem = "lesion_detector_%s_%s_imgsz%d" % (DATASET_VERSION, Path(MODEL).stem, IMGSZ)
    onnx, js = WORK / "export" / (stem + ".onnx"), WORK / "export" / (stem + ".json")
    assert onnx.exists() and js.exists(), "v2 export not found - run Step C first"
    APP_MODEL_DIR.mkdir(parents=True, exist_ok=True)
    for dst_name, src in (("lesion_detector_v1.onnx", onnx), ("lesion_detector_v1.json", js)):
        dst = APP_MODEL_DIR / dst_name
        if dst.exists():
            shutil.copy2(dst, dst.with_name(dst.stem + ".v1_backup" + dst.suffix))
        shutil.copy2(src, dst)
        print("copied", src.name, "->", dst)
    print("sha256 of the app model:", hashlib.sha256((APP_MODEL_DIR / "lesion_detector_v1.onnx").read_bytes()).hexdigest()[:16])
    print("Next: rebuild the app in Visual Studio; re-check the confidence thresholds in Settings.")

## Done - what to do with the results
- Send the Step D table to your documentation (it is also saved as `pipeline_logs/v1_vs_v2_test_comparison.csv`).
- The v2 run folder is `detector_work/runs/detector_v2_...`: curves, confusion matrix and `best.pt`.
- Add the v2 results, the draft precision/recall from Step A and the number of reviewed images to the methods notes.